In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging
)
import pandas as pd
import re
import json
from datetime import datetime
from datasets import Dataset
from huggingface_hub import login
from transformers import AutoConfig, AutoModel
import torch
from peft import PeftModel, PeftConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle
from huggingface_hub import login
import os
# from huggingface_hub import notebook_login
# notebook_login()

# access_token = os.environ["put your hf token"]
# login(token=access_token)


login(token="put your hf token")

# ########## Data Preparation #############
# Assign variables
base_path = "/root/pj_llm/dataset/preprocessed/pkl/"
raw_filename = "wholespine_ori_question.pkl"

# Set models
base_model = "ProbeMedicalYonseiMAILab/medllama3-v20"           # Hugging Face Basic Model
new_model = "ProbeMedicalYonseiMAILab/medllama3-v20-finetuned"  # Fine tuning Model

with open(base_path + raw_filename, 'rb') as f:
    raw_dataset = pickle.load(f)

def create_text_column(example):
    text = f"### Instruction:\n{example['instruction']}\n\nInput:\n{example['input']}\n\n### Response:\n{example['output']}"
    return text

raw_dataset['text'] = raw_dataset.apply(create_text_column, axis=1)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # 일관성을 위해 처음부터 설정
EOS_TOKEN = tokenizer.eos_token

def prompt_eos(example):
    example['text'] = example['text'] + EOS_TOKEN
    return example

raw_dataset = raw_dataset.apply(prompt_eos, axis=1)

# Divide into train, eval and test
'''
The Ratio of
train dataset: 70% / 80%
eval dataset:  15% / 10%
test dataset:  15% / 10%
'''
# Split data to train/eval/test
train_dataset, temp_dataset = train_test_split(raw_dataset, test_size=0.30, stratify=raw_dataset['output'], random_state=42)
eval_dataset, test_dataset = train_test_split(temp_dataset, test_size=0.50, stratify=temp_dataset['output'], random_state=42)
# train_dataset, temp_dataset = train_test_split(raw_dataset, test_size=0.2, stratify=raw_dataset['output'], random_state=42)
# eval_dataset, test_dataset = train_test_split(temp_dataset, test_size=0.5, stratify=temp_dataset['output'], random_state=42)
# Reset index
train_dataset.reset_index(drop=True, inplace=True)
eval_dataset.reset_index(drop=True, inplace=True)
test_dataset.reset_index(drop=True, inplace=True)

test_dataset = test_dataset.drop(['instruction', 'text', 'output'], axis=1)

# sequence_length 1024, 4bit quantization full-finetuning result data
df_1 = pd.read_csv("/root/pj_llm/codes/04_hallucination/before_pe_result/before_pe.csv")
new_df = pd.concat([df_1, test_dataset], axis=1)
new_df['report'] = new_df['input']
new_df = new_df.drop(['input'], axis=1)


df_2 = pd.read_csv("/root/pj_llm/codes/04_hallucination/after_pe_result/after_pe.csv") #after pe
new_df2 = pd.concat([df_2, test_dataset], axis=1)
new_df2['report'] = new_df2['input']
new_df2 = new_df2.drop(['input'], axis=1)

########## PEFT + BASE_MODEL ##############
base_model_name = "ProbeMedicalYonseiMAILab/medllama3-v20"
adapter_model_name = "/root/pj_llm/codes/04_hallucination/model_jh"

# Check GPU architecture
if torch.cuda.get_device_capability()[0] >= 8:
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# Set quantization with BitsAndBytesConfig
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

new_model = PeftModel.from_pretrained(model, adapter_model_name) #파라미터 합치는 코드
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True) #토크나이저는 베이스모델것과 동일하므로 상관없음


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
new_df2.head()

,correct,predicted,accuracy,precision,recall,f1_score,report
0,romets,romets,0.952381,0.954167,0.95207,0.95171,\n\nClinical information : Hepatocellular carc...
1,improved,improved,NaN,NaN,NaN,NaN,\nExam : Whole spine MRI with contrast enhance...
2,romets,romets,NaN,NaN,NaN,NaN,Exam : Whole spine MRI with contrast enhanceme...
3,stable,stable,NaN,NaN,NaN,NaN,EXAM: Whole spine MRI with contrast enhancemen...
4,romets,romets,NaN,NaN,NaN,NaN,\nExam: Whole spine MRI with contrast enhancem...


In [7]:
after_pe_errors = new_df2[new_df2.correct != new_df2.predicted]

In [8]:
after_pe_errors = after_pe_errors[['correct', 'predicted', 'report']]

In [9]:
after_pe_errors.head()

,correct,predicted,report
24,stable,romets,\nExam: Whole spine MRI with contrast enhancem...
26,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
65,progression,romets,\n\nEXAM : Whole spine MRI with contrast enhan...
70,romets,improved,Exam : Whole spine MRI with contrast enhanceme...
93,romets,improved,EXAM : Whole spine MRI with contrast enhanceme...


# Generator function

In [157]:
def generate_custom_response(prompt, max_new_tokens=300):
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Move input tensors to the same device as the model (usually CUDA)
    inputs = {key: value.to(new_model.device) for key, value in inputs.items()}
    
    outputs = new_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, top_p=0.9, temperature=0.1)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the portion of the response after "### response ###"
    response_start = "### explanation ###"
    response = response.split(response_start)[-1].strip()

    response = response.split("\n \n")[0].strip()    
    return response

## 왜 romets를 improved라고 분류했니?

In [13]:
print(after_pe_errors['report'].loc[26])

EXAM: Whole spine MRI with contrast enhancement and metal artifact reduction.
Clinical information: Upper limb weakness, s/p decompressive laminectomy and posterior fixation, s/p corpectomy and anterior interbody fixation due pathologic fracture T2 (Osteosarcoma), s/p Radiotherapy.
Findings:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Swelling and T2 hyperintensity with enhancement in posterior back muscle and subcutaneous fat layer of thoracic level.
• Epidural extension and encroachment of both foraminal canal at T2/3.
Bone marrow signal change and enhancement in T2 and T3.
Diffuse marrow signal changes including thoracic vertebrae.
• Clinical significance indeterminate.
Conclusions:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Decreased epidural extent compared with 2021-12-16 MRI.
R/O Remained lesion, T2 and T3.
Recommendation:
Clinical co

In [166]:
#test1
y_i = "improved"

report = """
EXAM: Whole spine MRI with contrast enhancement and metal artifact reduction.
Clinical information: Upper limb weakness, s/p decompressive laminectomy and posterior fixation, s/p corpectomy and anterior interbody fixation due pathologic fracture T2 (Osteosarcoma), s/p Radiotherapy.
Findings:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Swelling and T2 hyperintensity with enhancement in posterior back muscle and subcutaneous fat layer of thoracic level.
• Epidural extension and encroachment of both foraminal canal at T2/3.
Bone marrow signal change and enhancement in T2 and T3.
Diffuse marrow signal changes including thoracic vertebrae.
• Clinical significance indeterminate.
Conclusions:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Decreased epidural extent compared with 2021-12-16 MRI.
R/O Remained lesion, T2 and T3.
Recommendation:
Clinical correlation and follow up.
"""


prompt = f"""
### instruction ###
You are a radiologist. You have been assigned an important task: to generate explanation of the label you predicted based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this command and radiology report below:

### command ###
Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.

- Description of six labels
    - no
        - This label indicates the absence of any detectable abnormality or condition in the imaging study.
        - For example, 'no metastasis' means that the imaging did not reveal any evidence of cancer spreading to other parts of the body.
    - mets
        - Metastasis refers to the spread of cancer cells from the primary site to other parts of the body.
        - This label is used when imaging shows evidence of such spread, which is critical for staging and treatment planning.
    - progression
        - The term 'progression' is used when there is evidence that the disease is worsening or advancing.
        - In the context of cancer, this means the tumor has grown in size or spread further since the last imaging study.
    - stable
        - 'Stable' indicates that there has been no significant change in the condition or findings since the previous imaging study.
        - This means that the disease has neither progressed nor improved, remaining unchanged.
    - improved
        - This label signifies that there has been a positive change or reduction in the severity of the disease or abnormality.
        - For instance, in cancer, it means the tumor size has decreased or the extent of the disease has lessened compared to prior imaging.
    - romets
        - 'Rule out metastasis' is used when further investigation is needed to determine if metastasis is present.
        - It indicates that additional diagnostic tests are required to confirm or exclude the presence of metastatic disease.
        - R/O should be interpreted as romets.

Focus on the key terms in the "Conclusion" and "Impression" section
1. Focus on the "Conclusion" section for primary classification.
2. If needed, refer to the "Findings" section for additional context.
3. Ignore sections that do not contribute to the classification, such as patient history.
4. Pay special attention to key terms indicating the patient's condition, such as "R/O", "stable," "metastasis," "progression," "no evidenfinal," "improved," and similar phrases.
5. Provide final_the classification output only.

Let's think step by step.

### report ###
{report}


### response format ###.
Based on the predicted label and the report, provide the explanation why you predicted {y_i} based on the report.
Please write explanation within 300 characters with the keywords that you attended to and the part where the keyword is located : Conclusion, Impression, Findings, Recommendation, Exam
Generate your explanation within 300 characters.

### acknowledgement ###
Remember that the label you have predicted is {y_i}.
DO NOT repeat the response you already said. and do not mention the instruction and report after your response.
Generate explanation within 300 characters.

### explanation ###

"""

response = generate_custom_response(prompt)
print(f"""
Explanation:

{response}""")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Explanation:

Improved epidural extent compared with 2021-12-16 MRI.
R/O Remained lesion, T2 and T3.


In [14]:
print(after_pe_errors['report'].loc[70])

Exam : Whole spine MRI with contrast enhancement.  
Clinical information : RCC with multiple metastasis.      
COMPARISON : 2021-04-30 MRI.  
  
Findings :
Newly developed marrow replacing enhancing nodular lesion in the upper endplate of L2 right body. Lesion shows peripheral enhanement with internal  nonenhancement, peripheral low signal intensity rim. Schemorl`s node is suggested rather than metastasis.

Decreased peripheral enhancement of multiple marrow replacing lesions in T2 and T4, L4, right ilium and left femoral neck within scanned range.
No change of fatty marrow change in upper T- and lower L-spine to sacrum, suggestive of RTx-induced change.
 
 
Conclusion :
1. Newly devloped lesion, L2 body.
    - R/O Schmorl`s node
      - DDx. Metastasis.
2. Decreased enhancement of known multiple bone metastasis.


Recommendation : 
Clinical correlation and MRI short term follow up.


In [165]:
#test1
y_i = "improved"

report = """
Exam : Whole spine MRI with contrast enhancement.  
Clinical information : RCC with multiple metastasis.      
COMPARISON : 2021-04-30 MRI.  
  
Findings :
Newly developed marrow replacing enhancing nodular lesion in the upper endplate of L2 right body. Lesion shows peripheral enhanement with internal  nonenhancement, peripheral low signal intensity rim. Schemorl`s node is suggested rather than metastasis.

Decreased peripheral enhancement of multiple marrow replacing lesions in T2 and T4, L4, right ilium and left femoral neck within scanned range.
No change of fatty marrow change in upper T- and lower L-spine to sacrum, suggestive of RTx-induced change.
 
 
Conclusion :
1. Newly devloped lesion, L2 body.
    - R/O Schmorl`s node
      - DDx. Metastasis.
2. Decreased enhancement of known multiple bone metastasis.


Recommendation : 
Clinical correlation and MRI short term follow up.
"""


prompt = f"""
### instruction ###
You are a radiologist. You have been assigned an important task: to generate explanation of the label you predicted based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this command and radiology report below:

### command ###
Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.

- Description of six labels
    - no
        - This label indicates the absence of any detectable abnormality or condition in the imaging study.
        - For example, 'no metastasis' means that the imaging did not reveal any evidence of cancer spreading to other parts of the body.
    - mets
        - Metastasis refers to the spread of cancer cells from the primary site to other parts of the body.
        - This label is used when imaging shows evidence of such spread, which is critical for staging and treatment planning.
    - progression
        - The term 'progression' is used when there is evidence that the disease is worsening or advancing.
        - In the context of cancer, this means the tumor has grown in size or spread further since the last imaging study.
    - stable
        - 'Stable' indicates that there has been no significant change in the condition or findings since the previous imaging study.
        - This means that the disease has neither progressed nor improved, remaining unchanged.
    - improved
        - This label signifies that there has been a positive change or reduction in the severity of the disease or abnormality.
        - For instance, in cancer, it means the tumor size has decreased or the extent of the disease has lessened compared to prior imaging.
    - romets
        - 'Rule out metastasis' is used when further investigation is needed to determine if metastasis is present.
        - It indicates that additional diagnostic tests are required to confirm or exclude the presence of metastatic disease.
        - R/O should be interpreted as romets.

Focus on the key terms in the "Conclusion" and "Impression" section
1. Focus on the "Conclusion" section for primary classification.
2. If needed, refer to the "Findings" section for additional context.
3. Ignore sections that do not contribute to the classification, such as patient history.
4. Pay special attention to key terms indicating the patient's condition, such as "R/O", "stable," "metastasis," "progression," "no evidenfinal," "improved," and similar phrases.
5. Provide final_the classification output only.

Let's think step by step.


### report ###
{report}


### response format ###.
Based on the predicted label and the report, provide the explanation why you predicted {y_i} based on the report.
Please write explanation within 300 characters with the keywords that you attended to and the part where the keyword is located : Conclusion, Impression, Findings, Recommendation, Exam
When writing explanation, make sure the sentences are contextually connected to each other.
Generate explanation without unnecessary repetition.


### acknowledgement ###
Remember that the label you have predicted is {y_i}.
Generate explanation without unnecessary repetition.
Generate explanation which include only necessary information.


### explanation ###

"""

response = generate_custom_response(prompt)
print(f"""Explanation:

{response}""")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Explanation:

Improved status of multiple bone metastasis. 
Slightly decreased enhancement of known metastasis. 
No change of fatty marrow change in upper T- and lower L-spine to sacrum, suggestive of RTx-induced change. 
No evidence of central canal stenosis or neural foraminal stenosis. 
No evidence of disc herniation. 
No evidence of spinal cord compression.


In [15]:
print(after_pe_errors['report'].loc[93])

EXAM : Whole spine MRI with contrast enhancement. 
 
Clinical information : Lung cancer with brain, bone metastases. s/p intramedullary metastatic tumor removal (2022.2.7) s/p RTx on T11-L1. 
 
COMPARISON : Whole spine MRI taken on 2022.2.4. 
 
FINDINGS :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12 levels. 
T2 hyperintensity without enhancement in anterior aspect of conus medullaris at T11, T12 levels. 
- Focal pachymeningeal and leptomeningeal enhancement, subcutaneous T2 hyperintensity with enhancement at T11, T12 levels. 
No definite bone marrow replacing masses in whole spines. 
 
Central disc protrusion at T1/2, L2/3. 
Several enhancing small nodules in cerebellum and occipital lobe of cerebrum. 
R/O Brain metastases, refer to brain MRI taken on 2022.6.13. 
 
CONCLUSION :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12.
s/p Radiotherapy. 
- Focal pachymeningeal and leptomeningeal enhancement.
- Clinical significance i

In [163]:
#test1
y_i = "improved"

report = """
EXAM : Whole spine MRI with contrast enhancement. 
 
Clinical information : Lung cancer with brain, bone metastases. s/p intramedullary metastatic tumor removal (2022.2.7) s/p RTx on T11-L1. 
 
COMPARISON : Whole spine MRI taken on 2022.2.4. 
 
FINDINGS :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12 levels. 
T2 hyperintensity without enhancement in anterior aspect of conus medullaris at T11, T12 levels. 
- Focal pachymeningeal and leptomeningeal enhancement, subcutaneous T2 hyperintensity with enhancement at T11, T12 levels. 
No definite bone marrow replacing masses in whole spines. 
 
Central disc protrusion at T1/2, L2/3. 
Several enhancing small nodules in cerebellum and occipital lobe of cerebrum. 
R/O Brain metastases, refer to brain MRI taken on 2022.6.13. 
 
CONCLUSION :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12.
s/p Radiotherapy. 
- Focal pachymeningeal and leptomeningeal enhancement.
- Clinical significance indeterminate. Suggestive of post-treatment changes rather than recurrence. Follow-up is required. 
 
RECOMMENDATION : 
Clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. You have been assigned an important task: to generate explanation of the label you predicted based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this command and radiology report below:

### command ###
Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.

- Description of six labels
    - no
        - This label indicates the absence of any detectable abnormality or condition in the imaging study.
        - For example, 'no metastasis' means that the imaging did not reveal any evidence of cancer spreading to other parts of the body.
    - mets
        - Metastasis refers to the spread of cancer cells from the primary site to other parts of the body.
        - This label is used when imaging shows evidence of such spread, which is critical for staging and treatment planning.
    - progression
        - The term 'progression' is used when there is evidence that the disease is worsening or advancing.
        - In the context of cancer, this means the tumor has grown in size or spread further since the last imaging study.
    - stable
        - 'Stable' indicates that there has been no significant change in the condition or findings since the previous imaging study.
        - This means that the disease has neither progressed nor improved, remaining unchanged.
    - improved
        - This label signifies that there has been a positive change or reduction in the severity of the disease or abnormality.
        - For instance, in cancer, it means the tumor size has decreased or the extent of the disease has lessened compared to prior imaging.
    - romets
        - 'Rule out metastasis' is used when further investigation is needed to determine if metastasis is present.
        - It indicates that additional diagnostic tests are required to confirm or exclude the presence of metastatic disease.
        - R/O should be interpreted as romets.

Focus on the key terms in the "Conclusion" and "Impression" section
1. Focus on the "Conclusion" section for primary classification.
2. If needed, refer to the "Findings" section for additional context.
3. Ignore sections that do not contribute to the classification, such as patient history.
4. Pay special attention to key terms indicating the patient's condition, such as "R/O", "stable," "metastasis," "progression," "no evidenfinal," "improved," and similar phrases.
5. Provide final_the classification output only.

Let's think step by step.


### report ###
{report}


### response format ###.
Based on the predicted label and the report, provide the explanation why you predicted {y_i} based on the report.
Please write explanation within 300 characters with the keywords that you attended to with the part where the keyword is located : Conclusion, Impression, Findings, Recommendation, Exam
When writing explanation, make sure the sentences are contextually connected to each other.
Generate explanation without unnecessary repetition.

### acknowledgement ###
Remember that the label you have predicted is {y_i}.
Generate explanation without unnecessary repetition.
Generate explanation which include only necessary information.


### explanation ###
"""

response = generate_custom_response(prompt)
print(f"""Explanation:

{response}""")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Explanation:

s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12. 
s/p Radiotherapy. 
- Focal pachymeningeal and leptomeningeal enhancement.
- Clinical significance indeterminate. Suggestive of post-treatment changes rather than recurrence. Follow-up is required.


## 왜 stable을 romets라고 분류했니?

In [122]:
print(after_pe_errors['report'].loc[24])


Exam: Whole spine MRI with contrast enhancement. 
Clinical information: Breast cancer, Rt. Local recur, 
 
Compared with outside MRI taken on 2021.10.06  
 
Findings:   
No remarkable change of focal one marrow replacing and enhancing lesion in T9 vertebral body, R/O bone metastasis. 
No definite spinal cord signal change or enhancing lesion. 
No evidence of pathologic fracture in this study. 
No newly developed lesion. 
OPLL at T4-5-6.  
Disc bulging at L4/5.  
 
Impression: 
R/O Bone metastasis in T9 body without interval change. 
 
Recommendation:  
Clinical correlation.


In [159]:
y_i = "romets"

report = """
Exam: Whole spine MRI with contrast enhancement. 
Clinical information: Breast cancer, Rt. Local recur, 
 
Compared with outside MRI taken on 2021.10.06  
 
Findings:   
No remarkable change of focal one marrow replacing and enhancing lesion in T9 vertebral body, R/O bone metastasis. 
No definite spinal cord signal change or enhancing lesion. 
No evidence of pathologic fracture in this study. 
No newly developed lesion. 
OPLL at T4-5-6.  
Disc bulging at L4/5.  
 
Impression: 
R/O Bone metastasis in T9 body without interval change. 
 
Recommendation:  
Clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. You have been assigned an important task: to generate explanation of the label you predicted based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this command and radiology report below:

### command ###
Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.

- Description of six labels
    - no
        - This label indicates the absence of any detectable abnormality or condition in the imaging study.
        - For example, 'no metastasis' means that the imaging did not reveal any evidence of cancer spreading to other parts of the body.
    - mets
        - Metastasis refers to the spread of cancer cells from the primary site to other parts of the body.
        - This label is used when imaging shows evidence of such spread, which is critical for staging and treatment planning.
    - progression
        - The term 'progression' is used when there is evidence that the disease is worsening or advancing.
        - In the context of cancer, this means the tumor has grown in size or spread further since the last imaging study.
    - stable
        - 'Stable' indicates that there has been no significant change in the condition or findings since the previous imaging study.
        - This means that the disease has neither progressed nor improved, remaining unchanged.
    - improved
        - This label signifies that there has been a positive change or reduction in the severity of the disease or abnormality.
        - For instance, in cancer, it means the tumor size has decreased or the extent of the disease has lessened compared to prior imaging.
    - romets
        - 'Rule out metastasis' is used when further investigation is needed to determine if metastasis is present.
        - It indicates that additional diagnostic tests are required to confirm or exclude the presence of metastatic disease.
        - R/O should be interpreted as romets.

Focus on the key terms in the "Conclusion" and "Impression" section
1. Focus on the "Conclusion" section for primary classification.
2. If needed, refer to the "Findings" section for additional context.
3. Ignore sections that do not contribute to the classification, such as patient history.
4. Pay special attention to key terms indicating the patient's condition, such as "R/O", "stable," "metastasis," "progression," "no evidenfinal," "improved," and similar phrases.
5. Provide final_the classification output only.

Let's think step by step.


### report ###
{report}


### response format ###.
Based on the predicted label and the report, provide the explanation why you predicted {y_i} based on the report.
Please write explanation within 300 characters with the keywords that you attended to and the part where the keyword is located : Conclusion, Impression, Findings, Recommendation, Exam
When writing explanation, make sure the sentences are contextually connected to each other.
Generate explanation without unnecessary repetition.


### acknowledgement ###
Remember that the label you have predicted is {y_i}.
Generate explanation without unnecessary repetition.
Generate explanation which include only necessary information.


### explanation ###
"""

response = generate_custom_response(prompt)
print(f"""Explanation:

{response}""")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Explanation:

R/O Bone metastasis in T9 body without interval change. 
No remarkable change of focal one marrow replacing lesion in T9 vertebral body. 
No evidence of pathologic fracture in this study. 
No newly developed lesion.


## 왜 progression을 romets라고 분류했니?

In [131]:
print(after_pe_errors['report'].loc[65])



EXAM : Whole spine MRI with contrast enhancement 
HISTORY :  Prostate cancer with multiple bone metastases 
 
Compared to previous MRI taken on 2019.08.16, 
 
FINDING :  
Newly developed bone marrow replacing lesions with enhancement in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
Not well delineation of previous noted enhancing lesion in the lamina of C2. 
 
Spondylolisthesis of C4 over C5. 
Retrolisthesis of C6 over C7. 
Mild kyphotic deformity and mild central canal stenosis on C3-4-5-6-7 level. 
 
Disc bulging in T12/L1, L2/3, L3/4, and L4/5. 
- Mild central canal narrowing and bilateral neural foraminal narrowing in L3/4. 
- Bilateral moderate neural foraminal narrowing in L4/5. 
 
Spondylolytic spondylolisthesis of L5 over S1. 
- Bilateral severe neural foraminal narrowing. 
 
IMPRESSION :  
1. Probable bone metastases in T1 vertebral body, left lamina and pedicle of L5, left sacrum and left ilium.
2. Not well delineation of previous noted enhancing

In [158]:
y_i = "romets"

report = """
EXAM : Whole spine MRI with contrast enhancement 
HISTORY :  Prostate cancer with multiple bone metastases 
 
Compared to previous MRI taken on 2019.08.16, 
 
FINDING :  
Newly developed bone marrow replacing lesions with enhancement in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
Not well delineation of previous noted enhancing lesion in the lamina of C2. 
 
Spondylolisthesis of C4 over C5. 
Retrolisthesis of C6 over C7. 
Mild kyphotic deformity and mild central canal stenosis on C3-4-5-6-7 level. 
 
Disc bulging in T12/L1, L2/3, L3/4, and L4/5. 
- Mild central canal narrowing and bilateral neural foraminal narrowing in L3/4. 
- Bilateral moderate neural foraminal narrowing in L4/5. 
 
Spondylolytic spondylolisthesis of L5 over S1. 
- Bilateral severe neural foraminal narrowing. 
 
IMPRESSION :  
1. Probable bone metastases in T1 vertebral body, left lamina and pedicle of L5, left sacrum and left ilium.
2. Not well delineation of previous noted enhancing lesion in the lamina of C2. 
3. Spondylolytic spondylolisthesis of L5 over S1 with bilateral severe neural foraminal narrowing. 
 
RECOMMEND : clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. You have been assigned an important task: to generate explanation of the label you predicted based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this command and radiology report below:

### command ###
Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.

- Description of six labels
    - no
        - This label indicates the absence of any detectable abnormality or condition in the imaging study.
        - For example, 'no metastasis' means that the imaging did not reveal any evidence of cancer spreading to other parts of the body.
    - mets
        - Metastasis refers to the spread of cancer cells from the primary site to other parts of the body.
        - This label is used when imaging shows evidence of such spread, which is critical for staging and treatment planning.
    - progression
        - The term 'progression' is used when there is evidence that the disease is worsening or advancing.
        - In the context of cancer, this means the tumor has grown in size or spread further since the last imaging study.
    - stable
        - 'Stable' indicates that there has been no significant change in the condition or findings since the previous imaging study.
        - This means that the disease has neither progressed nor improved, remaining unchanged.
    - improved
        - This label signifies that there has been a positive change or reduction in the severity of the disease or abnormality.
        - For instance, in cancer, it means the tumor size has decreased or the extent of the disease has lessened compared to prior imaging.
    - romets
        - 'Rule out metastasis' is used when further investigation is needed to determine if metastasis is present.
        - It indicates that additional diagnostic tests are required to confirm or exclude the presence of metastatic disease.
        - R/O should be interpreted as romets.

Focus on the key terms in the "Conclusion" and "Impression" section
1. Focus on the "Conclusion" section for primary classification.
2. If needed, refer to the "Findings" section for additional context.
3. Ignore sections that do not contribute to the classification, such as patient history.
4. Pay special attention to key terms indicating the patient's condition, such as "R/O", "stable," "metastasis," "progression," "no evidenfinal," "improved," and similar phrases.
5. Provide final_the classification output only.

Let's think step by step.


### report ###
{report}


### response format ###.
Based on the predicted label and the report, provide the explanation why you predicted {y_i} based on the report.
Please write explanation within 300 characters with the keywords that you attended to and the part where the keyword is located : Conclusion, Impression, Findings, Recommendation, Exam
When writing explanation, make sure the sentences are contextually connected to each other.
Generate explanation without unnecessary repetition.


### acknowledgement ###
Remember that the label you have predicted is {y_i}.
Generate explanation without unnecessary repetition.
Generate explanation which include only necessary information.


### explanation ###
"""

response = generate_custom_response(prompt)
print(f"""Explanation:

{response}""")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Explanation:

R/O Bone metastasis in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
- Not well delineation of previous noted enhancing lesion in the lamina of C2. 
- Slightly increased extent of bone marrow replacing lesions in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
- Pathologic compression fracture in T1 vertebral body. 
- No evidence of central canal stenosis or neural foraminal stenosis.
